### Understanding SQL Agents

In [1]:
import os

from dotenv import load_dotenv
from langchain_classic.text_splitter import CharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.utilities import SQLDatabase

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set.")

temperature = 0.7
max_tokens = 1500
model_name = "gpt-3.5-turbo"

llm = ChatOpenAI(
    model=model_name,
    temperature=temperature,
    max_tokens=max_tokens,
    openai_api_key=openai_api_key
)

In [3]:
db_path = "sqlite:///./chinook.db"
db = SQLDatabase.from_uri(db_path)

print(db.dialect)
print(db.get_usable_table_names())

sqlite
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [4]:
from langchain_classic.chains import create_sql_query_chain

chain = create_sql_query_chain(
    llm=llm,
    db=db,
)

In [5]:
print(chain.get_prompts()[0].template)

You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: 

In [6]:
question = "How many employees are there in the database?"

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: How many employees are there in the database?
Response: SELECT COUNT("EmployeeId") as TotalEmployees
FROM "Employee"


In [7]:
question = "which country's customers have spent the most?"

chain = create_sql_query_chain(
    llm=llm,
    db=db,
)

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: which country's customers have spent the most?
Response: SELECT c.Country, SUM(i.Total) AS TotalSpent
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.Country
ORDER BY TotalSpent DESC
LIMIT 1;


In [8]:
question = "How many employees are there in the database?"

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: How many employees are there in the database?
Response: SELECT COUNT("EmployeeId") AS "TotalEmployees" FROM "Employee"


In [9]:
db.run(response)

'[(8,)]'

In [10]:
chain.get_prompts()[0].pretty_print()

You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result

In [12]:
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool
from langchain_core.tools import tool


@tool
def parse(query_string: str) -> str:
    """Parse the SQL query string and return a human-readable explanation."""

    splitted_string = query_string.split(":")

    if (len(splitted_string) >= 2):
        query = splitted_string[1].strip()
    else:
        query = query_string

    return query.strip()

### SQL Query Validation Tool

This tool validates SQL queries using Local Ollama to ensure:
1. The query is valid SQLite syntax
2. The query only contains SELECT statements (no INSERT, UPDATE, DELETE, DROP, etc.)

In [11]:
from langchain_ollama import ChatOllama

# Initialize Ollama LLM for query validation
validator_llm = ChatOllama(
    model="gpt-oss:latest",
    temperature=0.1,  # Lower temperature for more consistent validation
    max_tokens=500,
    base_url="http://localhost:11434"
)

In [13]:
@tool
def validate_sql_query(query_string: str) -> str:
    """
    Validate that the SQL query is valid SQLite and only contains SELECT statements.
    
    Args:
        query_string: The SQL query to validate
        
    Returns:
        The cleaned and validated query string
        
    Raises:
        ValueError: If the query is invalid or contains non-SELECT operations
    """
    # First, parse the query string (similar to parse function)
    splitted_string = query_string.split(":")
    if len(splitted_string) >= 2:
        query = splitted_string[1].strip()
    else:
        query = query_string.strip()
    
    # Remove extra whitespace and newlines
    query = " ".join(query.split())
    
    # Use Ollama to validate the query
    validation_prompt = f"""
You are a SQL validation expert. Analyze the following SQL query and determine:
1. Is it valid SQLite syntax?
2. Does it ONLY contain SELECT statements (no INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, etc.)?

SQL Query:
{query}

Respond ONLY in this exact format:
VALID: yes/no
SELECT_ONLY: yes/no
REASON: Brief explanation

Be strict: if the query contains ANY data modification or schema changes, SELECT_ONLY must be 'no'.
"""
    
    try:
        response = validator_llm.invoke(validation_prompt)
        validation_result = response.content
        
        # Parse the validation response
        lines = validation_result.strip().split('\n')
        is_valid = False
        is_select_only = False
        reason = "Unknown validation result"
        
        for line in lines:
            if line.startswith("VALID:"):
                is_valid = "yes" in line.lower()
            elif line.startswith("SELECT_ONLY:"):
                is_select_only = "yes" in line.lower()
            elif line.startswith("REASON:"):
                reason = line.replace("REASON:", "").strip()
        
        # Check validation results
        if not is_valid:
            raise ValueError(f"Invalid SQLite query: {reason}")
        
        if not is_select_only:
            raise ValueError(f"Query contains non-SELECT operations: {reason}")
        
        return query
        
    except Exception as e:
        if isinstance(e, ValueError):
            raise
        raise ValueError(f"Query validation failed: {str(e)}")

In [15]:
execute_query = QuerySQLDatabaseTool(db=db)
write_query = create_sql_query_chain(
    llm=llm,
    db=db,
)
chain = write_query | parse | validate_sql_query | execute_query
question = "How many employees are there in the database?"
response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: How many employees are there in the database?
Response: [(8,)]


In [16]:
question = "which country's customers have spent the most?"

execute_query = QuerySQLDatabaseTool(db=db)
write_query = create_sql_query_chain(
    llm=llm,
    db=db,
)
chain = write_query | parse | validate_sql_query | execute_query

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: which country's customers have spent the most?
Response: [('USA', 523.06)]


In [17]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [18]:
answer_prompt = PromptTemplate.from_template(
    """
        Given the following user question, corresponding SQL Query, and SQL Result, 
        answer the user question in a concise manner using business professional english and
        give specific details from the SQL Result with explanations.
        
        User Question: {question}
        SQL Query: {query}
        SQL Result: {result}
        Answer: 
    """
)

# This chain takes a user question, generates a SQL query, executes it, and produces a business-professional answer.
# Steps:
# 1. Passes the input through unchanged (RunnablePassthrough).
# 2. Assigns the generated SQL query to the "query" key using write_query.
# 3. Executes the SQL query:
#    - Takes the "query" output,
#    - Parses it for readability,
#    - Runs it against the database (execute_query),
#    - Assigns the result to the "result" key.
# 4. Formats the question, query, and result into a prompt using answer_prompt.
# 5. Sends the prompt to the language model (llm) for a concise, professional answer.
# 6. Parses the output as a string (StrOutputParser).

chain = RunnablePassthrough \
    .assign(query=write_query) \
    .assign(result=itemgetter("query") | parse | validate_sql_query | execute_query) | \
    answer_prompt | llm | StrOutputParser()

In [19]:
question = "How many employees are there in the database?"

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: How many employees are there in the database?
Response: There are 8 employees in the database.


In [20]:
question = "which country's customers have spent the most? Give me the details also of the amount."

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: which country's customers have spent the most? Give me the details also of the amount.
Response: The customers from the USA have spent the most, with a total amount of $523.06. This information is based on the SQL query that calculated the total amount spent by customers in each country and sorted it in descending order. The USA had the highest total amount spent compared to other countries.


In [21]:
import ast
import re


def query_as_list(database, query):
    """
    Execute a SQL query and return the results as a list of dictionaries.
    """
    result = database.run(query)
    result = [el for sub in ast.literal_eval(result) for el in sub if el]
    result = [re.sub(r"\b\d+\b", "", string).strip() for string in result]

    return list(set(result))

artists = query_as_list(db, "SELECT Name FROM Artist")
albums = query_as_list(db, "SELECT Title FROM Album")

In [22]:
artists[5]

'Lulu Santos'

In [23]:
albums[:5]

['By The Way',
 'Meus Momentos',
 'Temple of the Dog',
 'Come Taste The Band',
 'Mezmerize']

In [24]:
from langchain_classic.agents.agent_toolkits import create_retriever_tool
from langchain_community.vectorstores import FAISS

In [25]:
vector_database = FAISS.from_texts(
    texts=artists + albums,
    embedding=OpenAIEmbeddings(openai_api_key=openai_api_key),
)

retriever = vector_database.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

description = """
    use to lookup values to filter on.
    input is an approximate spelling of the valid and proper nouns.
    Use the noun most similar to the input.
    If the input is not a valid noun, return an empty string.
"""

retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="retriever",
    description=description,
)

In [26]:
response = retriever_tool.invoke("Alis Chains")

print("Input: Alis Chains")
print("Response:", response)

Input: Alis Chains
Response: Alice In Chains

Aisha Duo

Alanis Morissette


In [27]:
response = retriever_tool.invoke("Do we have any artists by named Alis Chains")

print("Input: Alis Chains")
print("Response:", response)

Input: Alis Chains
Response: Alice In Chains

Alanis Morissette

Various Artists


In [28]:
response = retriever_tool.invoke("Do we have any albums by named The last concert")

print("Input: The last concert")
print("Response:", response)

Input: The last concert
Response: The Final Concerts (Disc )

MK III The Final Concerts [Disc ]

The Last Night of the Proms
